# Phase 6 & 7: Threshold Tuning + SHAP Explainability

**Goals:**
1. Re-train the best XGBoost configuration locally (no SageMaker needed here — just the final model)
2. Understand the **precision-recall tradeoff** through threshold selection
3. Final evaluation on the **held-out test set** (used once, at the very end)
4. Use **SHAP** to explain model predictions — what features drive individual fraud flags?

**Why threshold tuning matters:**  
A model outputs a probability like 0.73. "Is this fraud?" depends entirely on where you draw the line. There is no universally correct threshold — it is a **business decision**:
- Block at 0.3 → catch more fraud, frustrate more legitimate customers
- Block at 0.8 → fewer false positives, miss more real fraud

Your job is to make this tradeoff explicit.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
import shap

from sklearn.metrics import (
    precision_recall_curve, average_precision_score,
    roc_auc_score, confusion_matrix, classification_report
)
from src.preprocessing import load_data, scale_features, split_data

%matplotlib inline
shap.initjs()

## 1. Re-train Best XGBoost Model Locally

In [ ]:
df = load_data('../data/raw/creditcard.csv')
df = scale_features(df)
X_train, X_val, X_test, y_train, y_val, y_test = split_data(df)

# Compute scale_pos_weight from training data
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
spw = neg / pos
print(f'scale_pos_weight: {spw:.2f}')

In [ ]:
model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=spw,
    subsample=0.8,
    colsample_bytree=0.8,
    use_label_encoder=False,
    eval_metric='aucpr',
    random_state=42,
)

model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=50,
)

y_prob_val = model.predict_proba(X_val)[:, 1]
print(f'\nValidation AUC-PR: {average_precision_score(y_val, y_prob_val):.4f}')

## 2. Precision-Recall Tradeoff Visualization

In [ ]:
prec, rec, thresholds = precision_recall_curve(y_val, y_prob_val)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# PR curve
auc_pr = average_precision_score(y_val, y_prob_val)
axes[0].plot(rec, prec, color='steelblue', lw=2)
axes[0].set_xlabel('Recall (fraction of fraud caught)')
axes[0].set_ylabel('Precision (fraction of flags that are real fraud)')
axes[0].set_title(f'Precision-Recall Curve (AUC-PR={auc_pr:.3f})')
axes[0].axhline(y=y_val.mean(), color='gray', linestyle='--', label='Random')
axes[0].legend()

# Precision and Recall vs Threshold
axes[1].plot(thresholds, prec[:-1], label='Precision', color='steelblue')
axes[1].plot(thresholds, rec[:-1], label='Recall', color='crimson')
axes[1].set_xlabel('Decision Threshold')
axes[1].set_ylabel('Score')
axes[1].set_title('Precision & Recall vs Decision Threshold')
axes[1].legend()
# Mark some candidate thresholds
for t in [0.3, 0.5, 0.7]:
    axes[1].axvline(x=t, color='gray', linestyle=':', alpha=0.6)
    axes[1].text(t + 0.01, 0.5, str(t), fontsize=9, color='gray')

plt.tight_layout()
plt.savefig('../data/threshold_analysis.png', dpi=120)
plt.show()

## 3. Business-Driven Threshold Selection

Two common strategies:
- **Fixed recall target** — e.g., "we must catch at least 90% of fraud"
- **Fixed precision target** — e.g., "at most 10% of our manual review queue should be legitimate"

In [ ]:
# Find threshold that achieves >= 90% recall
target_recall = 0.90
# prec/rec are sorted by decreasing threshold; find last index where recall >= target
eligible = np.where(rec[:-1] >= target_recall)[0]
if len(eligible) > 0:
    # highest precision among eligible thresholds
    best_idx = eligible[np.argmax(prec[eligible])]
    best_threshold = thresholds[best_idx]
    best_precision = prec[best_idx]
    best_recall = rec[best_idx]
    print(f'Threshold for ≥{target_recall*100:.0f}% recall: {best_threshold:.3f}')
    print(f'  → Precision at this threshold: {best_precision:.4f}')
    print(f'  → Recall at this threshold:    {best_recall:.4f}')
else:
    print(f'Model cannot achieve {target_recall*100:.0f}% recall')
    best_threshold = 0.5

## 4. Final Evaluation on Held-Out Test Set

This is the first and only time we touch the test set. This is a strict rule — if you tune on the test set, your reported metrics are optimistic and misleading.

In [ ]:
y_prob_test = model.predict_proba(X_test)[:, 1]
y_pred_test = (y_prob_test >= best_threshold).astype(int)

auc_roc_test = roc_auc_score(y_test, y_prob_test)
auc_pr_test  = average_precision_score(y_test, y_prob_test)

print('=== FINAL TEST SET RESULTS ===')
print(f'AUC-ROC: {auc_roc_test:.4f}')
print(f'AUC-PR:  {auc_pr_test:.4f}')
print(f'Threshold: {best_threshold:.3f}')
print()
print(classification_report(y_test, y_pred_test, target_names=['Legitimate', 'Fraud']))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred_test)
tn, fp, fn, tp = cm.ravel()

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(['Predicted Legit', 'Predicted Fraud'])
ax.set_yticklabels(['Actual Legit', 'Actual Fraud'])
for i in range(2):
    for j in range(2):
        ax.text(j, i, f'{cm[i, j]:,}', ha='center', va='center', fontsize=14,
                color='white' if cm[i, j] > cm.max() / 2 else 'black')
ax.set_title(f'Confusion Matrix (threshold={best_threshold:.2f})')
plt.colorbar(im)
plt.tight_layout()
plt.savefig('../data/confusion_matrix.png', dpi=120)
plt.show()

print(f'True Positives  (fraud caught):     {tp:,}')
print(f'False Negatives (fraud missed):     {fn:,}')
print(f'False Positives (legit flagged):    {fp:,}')
print(f'True Negatives  (legit cleared):    {tn:,}')

## 5. SHAP Explainability

SHAP (SHapley Additive exPlanations) explains **why** the model gave a particular prediction.  
Each feature gets a SHAP value: positive = pushed toward fraud, negative = pushed toward legitimate.

In [ ]:
explainer = shap.TreeExplainer(model)

# Compute SHAP values for the test set (can take a minute)
X_test_df = pd.DataFrame(X_test, columns=X_train.columns if hasattr(X_train, 'columns') else [f'V{i}' for i in range(X_test.shape[1])])
shap_values = explainer.shap_values(X_test_df)
print(f'SHAP values shape: {shap_values.shape}')

In [ ]:
# Global feature importance: which features matter most overall?
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test_df, plot_type='bar', show=False)
plt.title('Global Feature Importance (SHAP)')
plt.tight_layout()
plt.savefig('../data/shap_importance.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Beeswarm plot: feature value AND impact direction
# Red = high feature value, blue = low. Horizontal position = impact on fraud score.
plt.figure(figsize=(10, 7))
shap.summary_plot(shap_values, X_test_df, show=False)
plt.title('SHAP Feature Impact (beeswarm)')
plt.tight_layout()
plt.savefig('../data/shap_beeswarm.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Explain a single fraud transaction — why did the model flag this one?
fraud_indices = np.where(y_test == 1)[0]
sample_idx = fraud_indices[0]  # first confirmed fraud in test set

print(f'Sample index: {sample_idx}, true label: Fraud')
print(f'Model probability: {y_prob_test[sample_idx]:.4f}')

shap.waterfall_plot(
    shap.Explanation(
        values=shap_values[sample_idx],
        base_values=explainer.expected_value,
        data=X_test_df.iloc[sample_idx],
    ),
    show=True
)

In [ ]:
# Explain a false positive — legitimate transaction flagged as fraud
fp_indices = np.where((y_test == 0) & (y_pred_test == 1))[0]
if len(fp_indices) > 0:
    sample_idx = fp_indices[0]
    print(f'False Positive index: {sample_idx}')
    print(f'Model probability: {y_prob_test[sample_idx]:.4f} (threshold: {best_threshold:.3f})')
    
    shap.waterfall_plot(
        shap.Explanation(
            values=shap_values[sample_idx],
            base_values=explainer.expected_value,
            data=X_test_df.iloc[sample_idx],
        ),
        show=True
    )
else:
    print('No false positives in test set at this threshold.')

## 6. Project Summary

| Phase | Notebook | Key Learning |
|---|---|---|
| EDA | 01_eda | Class imbalance severity; V4/V11/V14/V17 most discriminative |
| Feature Eng + Baseline | 02_baseline | Accuracy is misleading; AUC-PR is the right metric; class weights help |
| Tree Models | 03_tree_models | XGBoost > RF on AUC-PR; `scale_pos_weight` is critical |
| Anomaly Detection | 04_anomaly_detection | Unsupervised RCF useful for cold-start; supervised models are far better when labels exist |
| Threshold + SHAP | 05_threshold_tuning | Threshold is a business decision; SHAP explains individual predictions |

**Next ideas to extend this project:**
- Add hyperparameter optimization with SageMaker Automatic Model Tuning (Bayesian search)
- Register the best model in SageMaker Model Registry
- Stream new transactions through a SageMaker real-time endpoint
- Use SageMaker Model Monitor to detect data drift in production